In [15]:
import sys, os
mods_to_remove = [k for k in sys.modules if k.startswith('src')]
for m in mods_to_remove:
    del sys.modules[m]
print(f"✅ {len(mods_to_remove)} modules vidés")

✅ 10 modules vidés


In [16]:
sys.path.append(os.path.abspath(os.path.join('..')))

from src.ingestion.pdf_reader      import PDFReader
from src.preprocessing.cleaner     import DocumentCleaner
from src.preprocessing.segmenter   import DocumentSegmenter
from src.ai.llm_service            import LLMOrchestrator
from src.models.extraction_schemas import ExtractedCandidate

print("✅ Modules chargés")

✅ Modules chargés


In [17]:
OLLAMA_URL = "http://localhost:11434/api/generate"
CV_PATH    = "../data/raw/cvs/257264722 CV.pdf"
TR_PATH    = "../data/raw/transcripts/257264722 T .pdf"
CANDIDATE  = "Esra Al-Nashash"

In [18]:
reader  = PDFReader()
cleaner = DocumentCleaner()

cv_doc = reader.extract(CV_PATH)
tr_doc = reader.extract(TR_PATH)

print(f"CV — {len(cv_doc.content):>6} chars | {cv_doc.doc_type:<8} | "
      f"ocr_used={cv_doc.metadata['ocr_used']} | "
      f"ocr_available={cv_doc.metadata['ocr_available']} | "
      f"avg/page={cv_doc.metadata['avg_chars_per_page']}")

print(f"TR — {len(tr_doc.content):>6} chars | {tr_doc.doc_type:<8} | "
      f"ocr_used={tr_doc.metadata['ocr_used']} | "
      f"ocr_available={tr_doc.metadata['ocr_available']} | "
      f"avg/page={tr_doc.metadata['avg_chars_per_page']}")

print("\n--- CV 300 chars ---")
print(cv_doc.content[:300])
print("\n--- Transcript 300 chars ---")
print(tr_doc.content[:300])

CV —   3067 chars | native   | ocr_used=False | ocr_available=True | avg/page=1598.0
TR —  11521 chars | native   | ocr_used=False | ocr_available=True | avg/page=6567.7

--- CV 300 chars ---
[PAGE 1] Esra Jamil Al-Nashash Address Abu Dhabi, United Arab Emirates Phone +971 566723856 Email ealnashash@outlook.com Summary A graduate in Software Engineering with a Bachelor's degree and a Master's degree in Cybersecurity. Possesses a strong foundation in software development and a comprehensi

--- Transcript 300 chars ---
[PAGE 1] OFFICE OF THE REGISTRAR FICE School Code: 002785 Name: Esra AlNashash Birthdate (MM/DD): 02/20 Print Date: 10/29/2024 Student ID: N13301629 Institution ID: 002785 Page: 1 of 1 NYU Tandon School of Engineering Master of Science Major: Cybersecurity NYU Send To: ESRA JAMIL ALNASHASH NYU Appli


In [19]:
# 1. Vérifier la fin du Transcript (Là où se trouve souvent le Cumulative GPA)
print("--- TRANSCRIPT (Recherche du Cumulative GPA) ---")
print(tr_doc.content[::]) 

# 2. Vérifier le milieu du CV (Là où se trouvent les Publications)
print("\n---  CV (Recherche des Publications) ---")
print(cv_doc.content[::])

--- TRANSCRIPT (Recherche du Cumulative GPA) ---
[PAGE 1] OFFICE OF THE REGISTRAR FICE School Code: 002785 Name: Esra AlNashash Birthdate (MM/DD): 02/20 Print Date: 10/29/2024 Student ID: N13301629 Institution ID: 002785 Page: 1 of 1 NYU Tandon School of Engineering Master of Science Major: Cybersecurity NYU Send To: ESRA JAMIL ALNASHASH NYU Applied Cryptography CS-GY 6903 3.0 A Digital Forensics CS-GY 6963 3.0 B Application Security CS-GY 9163 3.0 A NYU Beginning of Graduate Record NYU AHRS EHRS QHRS QPTS GPA Current 9.0 9.0 9.0 33.000 3.667 Cumulative 27.0 27.0 27.0 103.002 3.815 NYU Summer 2024 Tandon School of Engineering Master of Science Major: Cybersecurity NYU Information Systems Security Engineering and Management CS-GY 6803 3.0 A NYU Degrees Awarded Master of Science 09/23/2024 Tandon School of Engineering Major: Cybersecurity Spring 2023 Tandon School of Engineering Master of Science Major: Cybersecurity NYU Information, Security and Privacy CS-GY 6813 3.0 A Computer Network

In [20]:
import sys, os, asyncio, json, time
sys.path.append(os.path.abspath(os.path.join('..')))

from src.ingestion.pdf_reader import PDFReader
from src.preprocessing.cleaner import DocumentCleaner
from src.ai.llm_service import LLMOrchestrator
from src.models.extraction_schemas import ExtractedCandidate

async def final_warrior_test():
    # 1. Configuration
    reader = PDFReader()
    cleaner = DocumentCleaner()
    # Note: On utilise localhost pour le test Notebook
    orchestrator = LLMOrchestrator(base_url="http://localhost:11434/api/generate")
    
    print("📋 Lecture et Nettoyage des documents de Esra...")
    cv_clean = cleaner.clean(reader.extract(CV_PATH).content)
    tr_clean = cleaner.clean(reader.extract(TR_PATH).content)

    print(f"🧠 Lancement des agents (Timeout: 300s)...")
    start = time.time()
    
    try:
        # Appel à la machine de guerre
        raw_results = await orchestrator.extract_parallel(
            cv_text=cv_clean, 
            tr_text=tr_clean,
            candidate_name="Esra Jamil Al-Nashash"
        )
        
        duration = round(time.time() - start, 2)
        print(f"✅ Extraction terminée en {duration}s")

        # Validation Pydantic (Le Garde-fou)
        candidate = ExtractedCandidate(**raw_results)

        print("\n" + "="*50)
        print("🏆 RÉSULTATS EXTRAITS PAR L'IA")
        print("="*50)
        print(f"👤 Nom : {candidate.full_name}")
        print(f"📍 Nationalité : {candidate.nationality}")
        print(f"🎓 BSc : {candidate.bsc_uni} | GPA: {candidate.bsc_gpa.raw_value or 'N/A'}")
        print(f"🎓 MSc : {candidate.msc_uni} | GPA: {candidate.msc_gpa.raw_value if candidate.msc_gpa else 'N/A'}")
        
        print(f"\n📚 PUBLICATIONS ({len(candidate.publications)})")
        for i, p in enumerate(candidate.publications, 1):
            print(f"   [{i}] {p.title[:70]}...")
            print(f"       Position: {p.author_position} / {p.total_authors} | Venue: {p.venue}")

    except Exception as e:
        print(f"❌ Erreur lors du test final : {e}")
        # En cas d'erreur, on affiche le JSON brut pour comprendre
        print("\n--- JSON BRUT REÇU (DEBUG) ---")
        print(json.dumps(raw_results, indent=2))
    finally:
        await orchestrator.close()

# Lancement
await final_warrior_test()

📋 Lecture et Nettoyage des documents de Esra...
🧠 Lancement des agents (Timeout: 300s)...
✅ Extraction terminée en 317.26s

🏆 RÉSULTATS EXTRAITS PAR L'IA
👤 Nom : Esra Jamil
📍 Nationalité : United Arab Emirati
🎓 BSc : None | GPA: N/A
🎓 MSc : New York University | GPA: 3.815

📚 PUBLICATIONS (1)
   [1] Enhancing IoT Security using Blockchain-Based Access Control Mechanism...
       Position: 1 / 1 | Venue: St Anne’s College, Oxford University, UK
